In [3]:
!pip install torch torchvision transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [4]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 21.5 MB/s eta 0:00:00


In [5]:
!pip install gradio pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.3 MB/s eta 0:00:00


In [21]:
import gradio as gr
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

class LocalQwenSupermarketAnalyzer:
    def __init__(self):
        self.model = None
        self.processor = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_loaded = False

    def load_model(self):
        """Load Qwen2-VL model locally"""
        try:
            print("Loading Qwen2-VL model... This may take a few minutes on first run.")

            # Load the model - using the smaller variant for better performance
            model_name = "Qwen/Qwen2-VL-2B-Instruct"

            self.model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                device_map="auto" if self.device == "cuda" else None,
                trust_remote_code=True
            )

            self.processor = AutoProcessor.from_pretrained(
                model_name,
                trust_remote_code=True
            )

            if self.device == "cpu":
                self.model = self.model.to("cpu")

            self.model_loaded = True
            print(f"Model loaded successfully on {self.device}")
            return "✅ Model loaded successfully!"

        except Exception as e:
            print(f"Error loading model: {str(e)}")
            return f"❌ Error loading model: {str(e)}"

    def process_image_input(self, image):
        """Process PIL image for Qwen model input"""
        if isinstance(image, str):
            # If image is a path
            image = Image.open(image).convert('RGB')
        elif hasattr(image, 'convert'):
            # If it's already a PIL image
            image = image.convert('RGB')

        return image

    def analyze_supermarket_image(self, image, analysis_type="comprehensive"):
        """Analyze supermarket shelf using local Qwen model"""

        if not self.model_loaded:
            load_result = self.load_model()
            if "Error" in load_result:
                return load_result

        # Define concise prompts for quick, high-level analysis
        prompts = {
            "comprehensive": """
            Analyze this supermarket shelf and provide a brief summary:

            **PRODUCTS:** List main product categories and brands visible
            **STOCK LEVELS:** Rate as High/Medium/Low for each category
            **EMPTY SPACES:** Note any gaps or low stock areas
            **RECOMMENDATIONS:** 2-3 key actions needed

            Keep response short and actionable - maximum 10 bullet points total.
            """,

            "inventory": """
            Provide quick inventory assessment:

            **STOCK STATUS:**
            - List product categories with stock levels (High/Medium/Low/Empty)
            - Identify which need immediate restocking

            **PRIORITIES:**
            - Priority 1: Urgent restocking needed
            - Priority 2: Monitor closely

            Keep response concise - focus on actionable items only.
            """,

            "visibility": """
            Quick visibility and placement analysis:

            **EYE-LEVEL:** What products are positioned optimally?
            **POOR VISIBILITY:** What needs better placement?
            **QUICK WINS:** 2-3 immediate improvements possible

            Provide brief, actionable recommendations only.
            """
        }

        try:
            # Process the image
            processed_image = self.process_image_input(image)

            # Create the conversation format for Qwen2-VL
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": prompts[analysis_type]}
                    ]
                }
            ]

            # Apply chat template
            text_prompt = self.processor.apply_chat_template(
                conversation,
                add_generation_prompt=True,
                tokenize=False
            )

            # Process inputs
            inputs = self.processor(
                text=text_prompt,
                images=processed_image,
                return_tensors="pt"
            )

            # Move to device
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            # Generate response with shorter output
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=256,  # Reduced from 1024 for shorter responses
                    temperature=0.3,     # Lower temperature for more focused output
                    do_sample=True,
                    pad_token_id=self.processor.tokenizer.pad_token_id
                )

            # Decode the response
            generated_text = self.processor.decode(
                output_ids[0][len(inputs["input_ids"][0]):],
                skip_special_tokens=True
            )

            return generated_text.strip()

        except Exception as e:
            return f"❌ Error during analysis: {str(e)}\n\nTroubleshooting tips:\n- Ensure image is clear and well-lit\n- Try a smaller image size\n- Check if model loaded properly"

def create_gradio_interface():
    """Create enhanced Gradio interface"""

    analyzer = LocalQwenSupermarketAnalyzer()

    def analyze_with_progress(image, analysis_type, progress=gr.Progress()):
        if image is None:
            return "⚠️ Please upload an image to analyze."

        progress(0.1, desc="Preparing analysis...")

        if not analyzer.model_loaded:
            progress(0.3, desc="Loading Qwen model (first time may take 2-3 minutes)...")
            load_result = analyzer.load_model()
            if "Error" in load_result:
                return load_result

        progress(0.7, desc="Analyzing shelf image...")
        result = analyzer.analyze_supermarket_image(image, analysis_type)

        progress(1.0, desc="Analysis complete!")
        return result

    # Custom CSS for better styling
    css = """
    .main-header {
        text-align: center;
        background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 20px;
        border-radius: 10px;
        margin-bottom: 20px;
    }
    .feature-box {
        border: 2px solid #e1e5e9;
        border-radius: 8px;
        padding: 15px;
        margin: 10px 0;
        background-color: #f8f9fa;
    }
    """

    with gr.Blocks(title="VK: Local Qwen Supermarket Analyzer", css=css, theme=gr.themes.Soft()) as demo:

        gr.HTML("""
        <div class="main-header">
            <h1>🛒 VK - AI Powered Supermarket Analysis</h1>
            <p>Free local analysis using Qwen2-VL Vision Model</p>
        </div>
        """)

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📤 Upload & Configure")

                image_input = gr.Image(
                    label="Upload Supermarket Shelf Image",
                    type="pil",
                    height=350
                )

                analysis_type = gr.Radio(
                    choices=[
                        ("🔍 Comprehensive Analysis", "comprehensive"),
                        ("📦 Inventory Focus", "inventory"),
                        ("👁️ Visibility & Placement", "visibility")
                    ],
                    label="Analysis Type",
                    value="comprehensive",
                    info="Choose your analysis focus"
                )

                analyze_btn = gr.Button(
                    "🚀 Start Analysis",
                    variant="primary",
                    size="lg",
                    scale=2
                )

                # gr.Markdown("""
                # <div class="feature-box">
                # <strong>💡 Tips for Best Results:</strong>
                # <ul>
                # <li>Use clear, well-lit images</li>
                # <li>Capture full shelf sections</li>
                # <li>Avoid blurry or angled shots</li>
                # <li>Include price tags if visible</li>
                # </ul>
                # </div>
                # """)

            with gr.Column(scale=2):
                gr.Markdown("### 📊 Analysis Results")

                output = gr.Markdown(
                    value="""
**Ready for Quick Analysis!** 🎯

Upload a shelf image for instant insights:

🔍 **Product Categories** - Quick identification
📊 **Stock Status** - High/Medium/Low levels
📈 **Action Items** - Priority restocking list
👀 **Placement Tips** - Quick optimization wins

*Concise, actionable results in seconds!*
                    """,
                    line_breaks=True
                )

        # Event handler
        analyze_btn.click(
            fn=analyze_with_progress,
            inputs=[image_input, analysis_type],
            outputs=output,
            show_progress=True
        )

        # Information sections
        with gr.Accordion("📋 Analysis Types Explained", open=False):
            gr.Markdown("""
            ### 🔍 Quick Overview
            Fast summary: products, stock levels, gaps, and key actions needed (max 10 points).

            ### 📦 Stock Check
            Inventory focus: High/Medium/Low stock status and restocking priorities.

            ### 👁️ Placement Check
            Visibility analysis: eye-level positioning and quick placement improvements.
            """)

        with gr.Accordion("⚙️ Installation Guide", open=False):
            gr.Markdown("""
            ### Step 1: Install Required Packages
            ```bash
            # Install PyTorch (choose based on your system)
            pip install torch torchvision torchaudio

            # Install Transformers and other dependencies
            pip install transformers
            pip install gradio pillow
            ```

            ### Step 2: Alternative Installation (if above fails)
            ```bash
            # Use conda instead
            conda install pytorch torchvision torchaudio -c pytorch
            pip install transformers gradio pillow
            ```

            ### Step 3: Run the Application
            ```bash
            python supermarket_analyzer.py
            ```

            ### System Requirements:
            - **RAM**: Minimum 8GB (16GB recommended)
            - **GPU**: Optional but recommended (CUDA compatible)
            - **Storage**: ~5GB for model download (first run)

            ### Troubleshooting:
            - If model loading fails, try running with `--trust-remote-code`
            - For memory issues, close other applications
            - GPU not required but speeds up processing significantly
            """)

        with gr.Accordion("🚀 Features & Benefits", open=False):
            gr.Markdown("""
            ### For Store Managers:
            ✅ **Real-time inventory monitoring**
            ✅ **Automated restocking alerts**
            ✅ **Product placement optimization**
            ✅ **Sales opportunity identification**

            ### Key Capabilities:
            🎯 **Product Recognition** - Identifies brands and categories
            📊 **Quantity Estimation** - Counts visible stock levels
            🔄 **Restocking Priorities** - Actionable recommendations
            📈 **Sales Optimization** - Placement and visibility insights

            ### Advantages of Local Processing:
            🔒 **Privacy**: Your images never leave your device
            💰 **Cost**: Completely free - no API charges
            ⚡ **Speed**: Fast processing after initial model load
            🌐 **Offline**: Works without internet connection
            """)

    return demo

# Launch the application
if __name__ == "__main__":
    print("🚀 Starting Local Qwen Supermarket Analyzer...")
    print("📥 Model will be downloaded on first run (~5GB)")
    print("⚡ Subsequent runs will be much faster!")
    print("💻 GPU detected!" if torch.cuda.is_available() else "💻 Running on CPU (GPU recommended for faster processing)")

    demo = create_gradio_interface()
    demo.launch(
        share=True,  # Set to True if you want to share publicly
        server_name="0.0.0.0",
        server_port=3000,
        show_error=True,
        inbrowser=True
    )

🚀 Starting Local Qwen Supermarket Analyzer...
📥 Model will be downloaded on first run (~5GB)
⚡ Subsequent runs will be much faster!
💻 GPU detected!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a7f1869137d6c9472.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
